# Week 18 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 18 Day 01: Stat-arb problem framing

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Learn stat-arb foundations including spreads, cointegration, and execution-aware controls.

## Continuity and Handoff
- Previous checkpoint: Week 17 Day 07: Portfolio Project
- Previous lesson file: content/week-17/day-07.md
- Today's deliverable: Implement spread builder with normalization options.
- Next handoff target: Week 18 Day 02: Cointegration basics
- Next lesson file: content/week-18/day-02.md

## Theory Concepts

### Concept 1: Relative-value logic
Relative-value logic is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Spread construction
Spread construction is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Stationarity requirement
Stationarity requirement is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Create a synthetic spread candidate and inspect mean behavior.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Stat-arb problem framing'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement spread builder with normalization options.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 18 Day 01, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Stat-arb problem framing' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Stat-arb problem framing as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Why is spread stationarity central for mean-reversion stat-arb?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 18 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(1801)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## Week 18 Day 01 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [3]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 119.000
price_t = 120.011
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.008496)
print('  gross_return_expected  =', 1.008496)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 119.0
  P_t    : 120.011
  r_t    : 0.008496 => 0.85%
  1+r_t  : 1.008496

Expected numeric check:
  simple_return_expected = 0.008496
  gross_return_expected  = 1.008496

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 18 Day 02: Cointegration basics

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Learn stat-arb foundations including spreads, cointegration, and execution-aware controls.

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 01: Stat-arb problem framing
- Previous lesson file: content/week-18/day-01.md
- Today's deliverable: Build cointegration screening utility for candidate pairs.
- Next handoff target: Week 18 Day 03: Signal generation and execution rules
- Next lesson file: content/week-18/day-03.md

## Theory Concepts

### Concept 1: Common trend vs mean-reverting residual
Common trend vs mean-reverting residual is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Cointegration testing intuition
Cointegration testing intuition is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Hedge ratio estimation
Hedge ratio estimation is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Estimate hedge ratio and test cointegration on synthetic pair data.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Cointegration basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build cointegration screening utility for candidate pairs.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 18 Day 02, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Cointegration basics' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Cointegration basics as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
How can structural breaks invalidate cointegration assumptions?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 18 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [4]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(1802)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## Week 18 Day 02 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [5]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 120.000
price_t = 121.080
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 120.0
  P_t    : 121.08
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Expected numeric check:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 18 Day 03: Signal generation and execution rules

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Learn stat-arb foundations including spreads, cointegration, and execution-aware controls.

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 02: Cointegration basics
- Previous lesson file: content/week-18/day-02.md
- Today's deliverable: Implement signal engine for pair spread strategy.
- Next handoff target: Week 18 Day 04: Backtest and cost-aware evaluation
- Next lesson file: content/week-18/day-04.md

## Theory Concepts

### Concept 1: Z-score entry and exit bands
Z-score entry and exit bands is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Position sizing for spreads
Position sizing for spreads is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Stop and timeout rules
Stop and timeout rules is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Generate spread z-score signals with practical exit logic.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Signal generation and execution rules'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement signal engine for pair spread strategy.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 18 Day 03, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Signal generation and execution rules' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Signal generation and execution rules as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Which exit rule reduces tail risk most effectively?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 18 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [6]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(1803)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## Week 18 Day 03 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [7]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 121.000
price_t = 122.150
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009504)
print('  gross_return_expected  =', 1.009504)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 121.0
  P_t    : 122.15
  r_t    : 0.009504 => 0.95%
  1+r_t  : 1.009504

Expected numeric check:
  simple_return_expected = 0.009504
  gross_return_expected  = 1.009504

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 18 Day 04: Backtest and cost-aware evaluation

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Learn stat-arb foundations including spreads, cointegration, and execution-aware controls.

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 03: Signal generation and execution rules
- Previous lesson file: content/week-18/day-03.md
- Today's deliverable: Add financing and slippage assumptions to spread backtest.
- Next handoff target: Week 18 Day 05: Failure modes and safeguards
- Next lesson file: content/week-18/day-05.md

## Theory Concepts

### Concept 1: Pair-level PnL decomposition
Pair-level PnL decomposition is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Borrow and financing assumptions
Borrow and financing assumptions is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Execution slippage sensitivity
Execution slippage sensitivity is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Backtest pair strategy with conservative cost assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Backtest and cost-aware evaluation'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Add financing and slippage assumptions to spread backtest.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 18 Day 04, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Backtest and cost-aware evaluation' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Backtest and cost-aware evaluation as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Where does hidden cost risk often get ignored in pairs strategies?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 18 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(1804)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## Week 18 Day 04 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [9]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 122.000
price_t = 123.220
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 122.0
  P_t    : 123.22
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Expected numeric check:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 18 Day 05: Failure modes and safeguards

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Learn stat-arb foundations including spreads, cointegration, and execution-aware controls.

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 04: Backtest and cost-aware evaluation
- Previous lesson file: content/week-18/day-04.md
- Today's deliverable: Implement basic safeguard rules and breach alerts.
- Next handoff target: Week 18 Day 06: Revision Sprint
- Next lesson file: content/week-18/day-06.md

## Theory Concepts

### Concept 1: Regime shift risk
Regime shift risk is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Correlation breakdown
Correlation breakdown is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Portfolio concentration control
Portfolio concentration control is a core part of 'Statistical arbitrage intuition'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Simulate strategy failure under abrupt regime change.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Failure modes and safeguards'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement basic safeguard rules and breach alerts.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 18 Day 05, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Failure modes and safeguards' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Failure modes and safeguards as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Which early-warning indicator should trigger pair deactivation?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 18 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [10]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(1805)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## Week 18 Day 05 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [11]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 123.000
price_t = 124.291
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010496)
print('  gross_return_expected  =', 1.010496)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 123.0
  P_t    : 124.291
  r_t    : 0.010496 => 1.05%
  1+r_t  : 1.010496

Expected numeric check:
  simple_return_expected = 0.010496
  gross_return_expected  = 1.010496

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 18 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 05: Failure modes and safeguards
- Previous lesson file: content/week-18/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 18 Day 07: Portfolio Project
- Next lesson file: content/week-18/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Re-run cointegration tests on alternate pairs
- Audit cost assumptions for realism
- Update safeguard checklist

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 18 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 18 Day 06: Revision Sprint
- Previous lesson file: content/week-18/day-06.md
- Today's deliverable: Pairs trading prototype
- Next handoff target: Week 19 Day 01: Agentic workflow design for research
- Next lesson file: content/week-19/day-01.md

## Project Title
Pairs trading prototype

## Problem Statement
Design a cost-aware stat-arb pairs framework with risk safeguards.

## Data Sources
- Candidate pair prices
- Spread diagnostics
- Cost assumptions

## Implementation Steps
1. Screen candidate pairs
2. Estimate spread and signal
3. Backtest with costs
4. Stress failure scenarios
5. Deliver go/no-go recommendation

## Evaluation Metrics
- Spread half-life proxy
- Net return
- Max drawdown
- Safeguard effectiveness

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
